In [ ]:
import pandas as pd
import numpy as np
import os
import subprocess
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import brotli
import io

In [ ]:
# run the rust code that generates the execution time data if the necessary files do not exist.
rust_project_dir = "../inc_search_eval"

data_dir = "inc_search_eval/paper_data" # This will generate plots using the raw data that generated the paper figures.
# data_dir = "inc_search_eval" # Uncomment this line to generate the data yourself. Once generated, the notebook wont overwrite it, you need to manually move/delete it to regenerate.

data_dir = data_dir.rstrip("/") + "/" # ensure it ends with a '/'


# These are the data sources needed for the plot
filenames = [
    data_dir+"results_nc_limits.csv.br",
]

# Check if ANY of the required files are missing
files_missing = any(not os.path.exists(f) for f in filenames)

if not files_missing:
    print("All result files already exist. Skipping Rust evaluation.")
else:
    # Ensure the target directory actually exists before running
    if not os.path.isdir(rust_project_dir):
        print(f"Error: Directory '{rust_project_dir}' not found.")
        sys.exit(1)

    print(f"Missing result files detected. Starting 'cargo run --release' in {rust_project_dir}...\n")

    try:
        # subprocess.run will wait for the command to finish.
        # It pipes standard output and error directly to your Python console.
        subprocess.run(
            ["cargo", "run", "--release"],
            cwd=rust_project_dir, # Sets the working directory for the command
            check=True            # Raises a CalledProcessError if the command fails
        )
        print("\nRust evaluation completed successfully.")
        
    except subprocess.CalledProcessError as e:
        print(f"\nExecution failed! Cargo exited with code {e.returncode}")
        sys.exit(e.returncode)
    except FileNotFoundError:
        print("\nError: 'cargo' command not found. Is Rust installed and in your system PATH?")
        sys.exit(1)

In [ ]:
def read_brotli_csv(path, **kwargs):
    """Reads a Brotli-compressed CSV file into a pandas DataFrame."""
    try:
        with open(path, 'rb') as f:
            return pd.read_csv(io.BytesIO(brotli.decompress(f.read())), **kwargs)
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return pd.DataFrame() # Returns an empty DataFrame without hardcoded columns


# Load the data
df = read_brotli_csv(data_dir+"results_nc_limits.csv.br", skipinitialspace=True)

def plot_heatmaps(df):
    df_filtered = df[~df['greedy_nc_limit'].isin([8,9])]

    # Positive values only for log scale calculation
    total_positive = df_filtered[df_filtered['total_avg_deviation'] > 0]['total_avg_deviation']
    avg_positive = df_filtered[df_filtered['avg_deviation'] > 0]['avg_deviation']

    # Shared vmin/vmax across both plots
    vmin = min(total_positive.min(), avg_positive.min())
    vmax = max(total_positive.max(), avg_positive.max())

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=False)
    fig.subplots_adjust(right=0.9)

    # Add a dedicated axis for the single colorbar on the right
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])

    def plot_custom_heatmap(pivot_data, ax, cbar=False, cbar_ax=None, cbar_kws=None):
        mask_zeros = (pivot_data == 0)
        
        # 1. Draw the Heatmap
        sns.heatmap(
            pivot_data,
            ax=ax,
            annot=True,
            fmt=".1e",
            cmap='viridis',
            cbar=cbar,
            cbar_ax=cbar_ax,
            cbar_kws=cbar_kws,
            norm=LogNorm(vmin=vmin, vmax=vmax),
            mask=mask_zeros
        )
        
        # grid iteration, add 0s when no data
        rows, cols = pivot_data.shape
        for y in range(rows):
            for x in range(cols):
                # Get value at this cell
                val = pivot_data.iloc[y, x]
                
                if val == 0:
                    rect = Rectangle((x, y), 1, 1, fill=True, color='black')
                    ax.add_patch(rect)
                    ax.text(x + 0.5, y + 0.5, '0', 
                            color='white', 
                            ha='center', va='center',
                            fontweight='normal')

    # Left
    pivot_total = df_filtered.pivot(
        index='full_nc_limit',
        columns='greedy_nc_limit',
        values='total_avg_deviation'
    ).sort_index(ascending=False)
    
    plot_custom_heatmap(pivot_total, axes[0])
    
    axes[0].set_xlabel('NC_LIMIT greedy', fontsize=14)
    axes[0].set_ylabel('NC_LIMIT non-greedy', fontsize=14)

    # Right
    pivot_avg = df_filtered.pivot(
        index='full_nc_limit',
        columns='greedy_nc_limit',
        values='avg_deviation'
    ).sort_index(ascending=False)

    plot_custom_heatmap(pivot_avg, axes[1], cbar=True, cbar_ax=cbar_ax, 
                        cbar_kws={'label': 'RI Deviation (Log Scale)'})
    
    axes[1].set_xlabel('NC_LIMIT greedy', fontsize=14)
    axes[1].set_ylabel('')

    # border
    for ax in axes:
        for _, spine in ax.spines.items():
            spine.set_visible(True)
            spine.set_linewidth(1.5)
            spine.set_edgecolor('black')
    for _, spine in cbar_ax.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_edgecolor('black')
    
    plt.savefig("Search_RI_deviation_over_NC_LIMITs_greedy_vs_non_greedy.png", dpi=300, bbox_inches="tight")
    plt.show()

plot_heatmaps(df)


# Table 2
The above image contains the Table 2 of the paper by observing the center diagonal of the left plot, and the diagonal with offset 1 above.

plot shows the redundancy information difference between greedy and non-greedy across various max ARQ-cycle limits. usually the more cycles are allowed, the more RI-efficient the search. The left plot shows the unconditional differences, the right shows the difference conditioned on it being non-zero. 